# Silver - Source Features (bronze vector flattening)

Bronze stores every DataBC WFS feature as opaque `geometry_json` / `properties_json`
strings. That is the right raw-fidelity choice for bronze, but it makes the layer
unusable for SQL and unusable for a Fabric data agent: neither can parse GeoJSON text
or reason about location.

This notebook flattens those features **once** into a curated, queryable table.

| | |
| --- | --- |
| **Reads (bronze)** | BC soil survey, soil project boundaries, quaternary geology, bedrock geology, fault lines |
| **Writes (silver)** | `silver_source_features` - one row per source feature, typed attributes, centroid, bbox, area, distance from AOI |
| | `silver_source_coverage` - one row per configured bronze table with status and record count |

`silver_source_features` is the table that lets the report agent answer questions like
*"what soil units underlie the AOI"*, *"how far is the nearest mapped fault"*, or
*"which surficial materials are within 2 km of the centre"* - none of which are
answerable from bronze as stored.

Attribute names differ between DataBC layers, so attributes are resolved by
**token-boundary pattern matching** against the actual property keys rather than by
hardcoded column names. A layer that changes its schema degrades to nulls in the
typed columns; `properties_json` is always retained verbatim.

Run with `silver_lakehouse` attached as the default lakehouse.

## Dependencies

Library dependencies are supplied by the **geohazard_env** Fabric Environment attached to this notebook, not by inline `%pip install`. Inline installation is disabled in many tenants and fails with MagicUsageError when a notebook runs from a pipeline. See `fabric/environment/requirements.txt`.

## 2. Parameters

Supplied by `pl_bronze_ingestion`. `PIPELINE_RUN_ID` scopes every written row so the
data agent can filter to a single screening run.

In [ ]:
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20
PIPELINE_RUN_ID = ""

## 3. Run identity, AOI, and projections

In [ ]:
import json
import re
import uuid
from datetime import datetime, timezone

from pyproj import Transformer
from shapely.geometry import Point
from shapely.geometry import shape as geometry_shape
from shapely.ops import transform as transform_geometry

LAT = float(LATITUDE)
LON = float(LONGITUDE)
RADIUS = float(RADIUS_KM)

if not -80.0 <= LAT <= 84.0:
    raise ValueError("LATITUDE must be between -80 and 84 degrees for UTM analysis.")
if not -180.0 <= LON <= 180.0:
    raise ValueError("LONGITUDE must be between -180 and 180 degrees.")
if not 0.0 < RADIUS <= 100.0:
    raise ValueError("RADIUS_KM must be greater than 0 and no more than 100 km.")

# Same run-id sanitisation as the gold notebook so artefacts line up across layers.
raw_run_id = str(PIPELINE_RUN_ID or "").strip()
if not raw_run_id:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    raw_run_id = f"manual-{timestamp}-{uuid.uuid4().hex[:8]}"
RUN_ID = re.sub(r"[^A-Za-z0-9._-]+", "-", raw_run_id)[:128].strip(".-_")
if not RUN_ID:
    raise ValueError("PIPELINE_RUN_ID did not contain any file-system-safe characters.")

AOI_NAME = f"AOI {LAT:.4f}, {LON:.4f}"
NOW_UTC = datetime.now(timezone.utc).isoformat()

utm_zone = min(60, max(1, int((LON + 180.0) // 6.0) + 1))
utm_epsg = (32600 if LAT >= 0 else 32700) + utm_zone
UTM_CRS = f"EPSG:{utm_epsg}"
_to_utm = Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True)
_to_wgs = Transformer.from_crs(UTM_CRS, "EPSG:4326", always_xy=True)
AOI_POINT_UTM = Point(*_to_utm.transform(LON, LAT))

print(f"run_id     : {RUN_ID}")
print(f"AOI        : {AOI_NAME}  (radius {RADIUS:g} km)")
print(f"Analysis CRS: {UTM_CRS}")

## 4. Bronze source registry

`vector` layers are flattened feature-by-feature. `catalog` layers are STAC metadata
tables - they are counted for source coverage but have no geometry to flatten.

In [ ]:
from pyspark.sql import functions as F

# Portable OneLake path: resolve immutable workspace and lakehouse IDs at runtime.
WS = notebookutils.runtime.context["currentWorkspaceId"]
BRONZE_LH_ID = notebookutils.lakehouse.get("bronze_lakehouse", workspaceId=WS).id
ONELAKE_ENDPOINT = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
BRONZE_ABFSS = f"abfss://{WS}@{ONELAKE_ENDPOINT}/{BRONZE_LH_ID}/Tables"

VECTOR_LAYERS = [
    {
        "table": "bronze_bc_soil_survey_polygons",
        "feature_class": "soil_survey",
        "source_name": "BC Soil Information Finder Tool (SIFT) - soil survey polygons",
        "score_soft": True,
    },
    {
        "table": "bronze_bc_soil_project_boundaries",
        "feature_class": "soil_project_boundary",
        "source_name": "BC SIFT - soil survey project boundaries",
        "score_soft": False,
    },
    {
        "table": "bronze_bc_quaternary_geology",
        "feature_class": "surficial_geology",
        "source_name": "BC quaternary / surficial geology",
        "score_soft": True,
    },
    {
        "table": "bronze_bc_bedrock_geology",
        "feature_class": "bedrock_geology",
        "source_name": "BC bedrock geology",
        "score_soft": False,
    },
    {
        "table": "bronze_bc_geological_faults",
        "feature_class": "fault",
        "source_name": "BC geological fault lines",
        "score_soft": False,
    },
]

CATALOG_LAYERS = [
    ("bronze_sentinel_2_l2a", "Sentinel-2 L2A optical STAC items"),
    ("bronze_sentinel_1_rtc", "Sentinel-1 RTC radar STAC items"),
    ("bronze_sentinel_1_grd", "Sentinel-1 GRD radar STAC items"),
    ("bronze_cop_dem_glo_30", "Copernicus DEM GLO-30 STAC items"),
    ("bronze_esa_worldcover", "ESA WorldCover STAC items"),
    ("bronze_io_lulc_9_class", "Esri 10 m land use / land cover STAC items"),
    ("bronze_alos_palsar_mosaic", "ALOS PALSAR mosaic STAC items"),
    ("bronze_hgb", "Harmonized global biomass STAC items"),
    ("bronze_satellite_stac_items", "Single-collection parameterised demo table"),
]

## 5. Attribute resolution

DataBC property keys vary by layer (`DRAINAGE`, `SOIL_DRAINAGE_CLASS`, ...). Patterns
are matched against **whole tokens** of each key, so `AGE` matches `AGE_PERIOD` but
never `DRAINAGE`. Exact key matches win over token matches.

In [ ]:
ATTRIBUTE_PATTERNS = {
    "name": ["SOILNAME_1", "SOILNAME", "SOIL_NAME", "MAP_UNIT_NAME", "UNIT_NAME",
             "FORMATION_NAME", "FORMATION"],
    "survey_name": ["PROJ_NAME", "PROJECT_NAME", "SURVEY_NAME"],
    "unit_code": ["SOILSYM_1", "MAP_UNIT_CODE", "SOIL_MAP_UNIT", "GEOLOGY_UNIT_CODE",
                  "UNIT_CODE", "SYMBOL", "CODE"],
    "drainage_class": ["DRAIN_1", "DRAINAGE_CLASS", "SOIL_DRAINAGE", "DRAINAGE"],
    "parent_material": ["MDEP_1", "PARENT_MATERIAL", "SURFICIAL_MATERIAL", "MODE_OF_DEPOSITION",
                        "DEPOSIT", "MATERIAL"],
    "texture": ["TEXTURE_1", "SURFACE_TEXTURE", "TEXTURE_CLASS", "TEXTURE"],
    "subgroup": ["DEV_1", "SOIL_SUBGROUP", "SUBGROUP", "GREAT_GROUP", "SOIL_ORDER", "CLASSIFICATION"],
    "rock_type": ["ROCK_TYPE_DESCRIPTION", "ROCK_TYPE", "ROCK_CLASS", "LITHOLOGY"],
    "age_period": ["STRATIGRAPHIC_AGE_NAME", "GEOLOGICAL_PERIOD", "AGE_PERIOD",
                   "PERIOD", "ERA", "STRAT_AGE", "AGE"],
    "fault_type": ["FAULT_TYPE", "FAULT_CLASS", "MOVEMENT_TYPE"],
    "description": ["DESCRIPTION", "DESCRIPTIVE_LEGEND", "REMARK", "COMMENT"],
}

# BC SIFT stores CODED attributes, not free text: DRAIN_1=W, MDEP_1=COLL, TEXTURE_1=SL.
# A keyword scan for words like "organic" or "poorly drained" matches none of them and
# silently returns the default for every polygon, which turns the survey signal into a
# flat constant. These tables decode the codes that actually appear in the data.
DRAIN_SCORE = {
    "VP": 1.00, "P": 0.90, "I": 0.60, "MW": 0.35, "W": 0.20, "R": 0.10, "VR": 0.05,
}
MDEP_SCORE = {
    "ORGA": 1.00, "ORG": 1.00, "LACU": 0.75, "FLLA": 0.72, "FLUV": 0.70,
    "MARI": 0.70, "GLMA": 0.70, "EOLI": 0.40, "COLL": 0.30, "MORA": 0.30,
    "BEDR": 0.05, "ROCK": 0.05,
}
TEXTURE_SCORE = {
    "C": 0.70, "SIC": 0.70, "SICL": 0.65, "CL": 0.60, "SC": 0.55, "SCL": 0.45,
    "SIL": 0.50, "SI": 0.50, "L": 0.40, "SL": 0.30, "LS": 0.25, "S": 0.20,
}
DRAIN_LABELS = {
    "VP": "very poorly drained", "P": "poorly drained", "I": "imperfectly drained",
    "MW": "moderately well drained", "W": "well drained", "R": "rapidly drained",
    "VR": "very rapidly drained",
}
MDEP_LABELS = {
    "ORGA": "organic", "ORG": "organic", "LACU": "lacustrine", "FLUV": "fluvial",
    "FLLA": "fluvio-lacustrine", "MARI": "marine", "GLMA": "glaciomarine",
    "EOLI": "eolian", "COLL": "colluvial", "MORA": "morainal", "BEDR": "bedrock",
    "ROCK": "bedrock",
}

# Free-text fallback for layers that are not SIFT (surficial geology, etc.).
SOFT_KEYS = {
    "very poorly": 1.0, "organic": 1.0, "peat": 1.0, "muck": 1.0, "bog": 1.0,
    "gleysol": 0.9, "gley": 0.9, "poorly drained": 0.9, "poorly": 0.85,
    "fluvial": 0.7, "alluv": 0.7, "lacustrine": 0.7, "marine": 0.7, "fen": 0.9,
    "imperfectly": 0.6, "clay": 0.6, "silt": 0.5,
}


def _sift_code(properties, key):
    value = properties.get(key)
    if value in (None, "", "-", "None"):
        return None
    return str(value).strip().upper()


def _component_score(properties, index):
    """Strongest soft-ground signal among drainage, parent material, and texture."""
    candidates = [
        DRAIN_SCORE.get(_sift_code(properties, "DRAIN_" + str(index))),
        MDEP_SCORE.get(_sift_code(properties, "MDEP_" + str(index))),
        TEXTURE_SCORE.get(_sift_code(properties, "TEXTURE_" + str(index))),
    ]
    present = [score for score in candidates if score is not None]
    return max(present) if present else None


def soft_soil_score_from_properties(properties):
    """Component-percent-weighted soft-ground score in [0,1].

    A SIFT polygon carries up to three soil components with a PERCENT_n share each, so
    the polygon score is the area-weighted mean of its components rather than a single
    lookup. Falls back to a keyword scan for non-SIFT layers.
    """
    total = 0.0
    weight = 0.0
    for index in (1, 2, 3):
        score = _component_score(properties, index)
        if score is None:
            continue
        percent = properties.get("PERCENT_" + str(index))
        try:
            share = float(percent) if percent not in (None, "", "None") else 0.0
        except (TypeError, ValueError):
            share = 0.0
        if share <= 0:
            share = 1.0
        total += score * share
        weight += share
    if weight > 0:
        return round(total / weight, 4)

    blob = " ".join(str(value) for value in properties.values()).lower()
    best = 0.0
    for keyword, value in SOFT_KEYS.items():
        if keyword in blob:
            best = max(best, value)
    return best if best > 0 else 0.4

MAX_ATTRIBUTE_CHARS = 512


def _tokens(text):
    return [token for token in re.split(r"[^A-Za-z0-9]+", text.upper()) if token]


def pick_attribute(properties, patterns):
    """Resolve one canonical attribute from arbitrary source property keys."""
    items = [(key, value) for key, value in properties.items()
             if value not in (None, "") and not isinstance(value, (dict, list))]
    if not items:
        return None
    upper_keys = {key: key.upper() for key, _ in items}
    key_tokens = {key: set(_tokens(key)) for key, _ in items}
    for pattern in patterns:
        target = pattern.upper()
        for key, value in items:
            if upper_keys[key] == target:
                return str(value)[:MAX_ATTRIBUTE_CHARS]
        pattern_tokens = _tokens(target)
        if not pattern_tokens:
            continue
        for key, value in items:
            if all(token in key_tokens[key] for token in pattern_tokens):
                return str(value)[:MAX_ATTRIBUTE_CHARS]
    return None




## 6. Flatten each vector layer

Geometry work happens in the local UTM zone so areas, lengths, and distances are in
metres. Centroids are computed in UTM and inverse-transformed, which is more faithful
than taking a centroid of unprojected degrees.

In [ ]:
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

FEATURE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("feature_key", StringType(), True),
    StructField("feature_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_layer", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("feature_class", StringType(), True),
    StructField("geometry_type", StringType(), True),
    StructField("centroid_lon", DoubleType(), True),
    StructField("centroid_lat", DoubleType(), True),
    StructField("bbox_minx", DoubleType(), True),
    StructField("bbox_miny", DoubleType(), True),
    StructField("bbox_maxx", DoubleType(), True),
    StructField("bbox_maxy", DoubleType(), True),
    StructField("area_km2", DoubleType(), True),
    StructField("length_km", DoubleType(), True),
    StructField("distance_from_aoi_km", DoubleType(), True),
    StructField("name", StringType(), True),
    StructField("survey_name", StringType(), True),
    StructField("unit_code", StringType(), True),
    StructField("drainage_class", StringType(), True),
    StructField("parent_material", StringType(), True),
    StructField("texture", StringType(), True),
    StructField("subgroup", StringType(), True),
    StructField("rock_type", StringType(), True),
    StructField("age_period", StringType(), True),
    StructField("fault_type", StringType(), True),
    StructField("description", StringType(), True),
    StructField("soft_soil_score", DoubleType(), True),
    StructField("aoi_name", StringType(), True),
    StructField("aoi_lat", DoubleType(), True),
    StructField("aoi_lon", DoubleType(), True),
    StructField("aoi_radius_km", DoubleType(), True),
    StructField("analysis_crs", StringType(), True),
    StructField("geometry_wkt", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("ingested_at_utc", StringType(), True),
])

COVERAGE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("feature_class", StringType(), True),
    StructField("status", StringType(), True),
    StructField("record_count", IntegerType(), True),
    StructField("note", StringType(), True),
    StructField("checked_at_utc", StringType(), True),
])


def read_bronze(table_name):
    return spark.read.format("delta").load(f"{BRONZE_ABFSS}/{table_name}")


def flatten_layer(config):
    """Return (feature_rows, coverage_row) for one bronze vector table."""
    table = config["table"]
    coverage = {
        "run_id": RUN_ID,
        "source_name": config["source_name"],
        "table_name": table,
        "feature_class": config["feature_class"],
        "status": "available",
        "record_count": 0,
        "note": None,
        "checked_at_utc": NOW_UTC,
    }
    try:
        source_rows = (read_bronze(table)
                       .select("feature_id", "source_layer", "geometry_json", "properties_json")
                       .collect())
    except Exception as error:
        coverage["status"] = "unavailable"
        coverage["record_count"] = None
        coverage["note"] = f"bronze table could not be read: {str(error).splitlines()[0][:200]}"
        print(f"  {table:<38} UNAVAILABLE")
        return [], coverage

    coverage["record_count"] = len(source_rows)
    if not source_rows:
        coverage["status"] = "empty"
        coverage["note"] = ("source returned no features over this AOI; treat as a data "
                            "gap, not a measured zero")
        print(f"  {table:<38} EMPTY (0 features)")
        return [], coverage

    rows = []
    skipped = 0
    for source_row in source_rows:
        try:
            geometry_dict = json.loads(source_row["geometry_json"]) if source_row["geometry_json"] else None
            if not geometry_dict:
                skipped += 1
                continue
            geometry = geometry_shape(geometry_dict)
            if geometry.is_empty:
                skipped += 1
                continue
            geometry_utm = transform_geometry(_to_utm.transform, geometry)
            if not geometry_utm.is_valid:
                geometry_utm = geometry_utm.buffer(0)

            centroid_utm = geometry_utm.centroid
            centroid_lon, centroid_lat = _to_wgs.transform(centroid_utm.x, centroid_utm.y)
            min_x, min_y, max_x, max_y = geometry.bounds

            area_km2 = float(geometry_utm.area) / 1_000_000.0
            length_km = float(geometry_utm.length) / 1000.0
            distance_km = float(geometry_utm.distance(AOI_POINT_UTM)) / 1000.0

            try:
                properties = json.loads(source_row["properties_json"]) if source_row["properties_json"] else {}
            except Exception:
                properties = {}
            if not isinstance(properties, dict):
                properties = {}

            attributes = {
                canonical: pick_attribute(properties, patterns)
                for canonical, patterns in ATTRIBUTE_PATTERNS.items()
            }
            # SIFT codes are unreadable to a report agent; carry decoded labels.
            if attributes.get("drainage_class"):
                attributes["drainage_class"] = DRAIN_LABELS.get(
                    attributes["drainage_class"].strip().upper(),
                    attributes["drainage_class"])
            if attributes.get("parent_material"):
                attributes["parent_material"] = MDEP_LABELS.get(
                    attributes["parent_material"].strip().upper(),
                    attributes["parent_material"])

            rows.append({
                "run_id": RUN_ID,
                "feature_key": f"{config['feature_class']}:{source_row['feature_id']}",
                "feature_id": source_row["feature_id"],
                "source_table": table,
                "source_layer": source_row["source_layer"],
                "source_name": config["source_name"],
                "feature_class": config["feature_class"],
                "geometry_type": geometry.geom_type,
                "centroid_lon": float(centroid_lon),
                "centroid_lat": float(centroid_lat),
                "bbox_minx": float(min_x),
                "bbox_miny": float(min_y),
                "bbox_maxx": float(max_x),
                "bbox_maxy": float(max_y),
                "area_km2": area_km2 if area_km2 > 0 else None,
                "length_km": length_km if length_km > 0 else None,
                "distance_from_aoi_km": distance_km,
                "soft_soil_score": (soft_soil_score_from_properties(properties)
                                    if config["score_soft"] else None),
                "aoi_name": AOI_NAME,
                "aoi_lat": LAT,
                "aoi_lon": LON,
                "aoi_radius_km": RADIUS,
                "analysis_crs": UTM_CRS,
                "geometry_wkt": geometry.wkt,
                "properties_json": source_row["properties_json"],
                "ingested_at_utc": NOW_UTC,
                **attributes,
            })
        except Exception:
            skipped += 1
            continue

    if skipped:
        coverage["note"] = f"{skipped} of {len(source_rows)} features had unusable geometry"
    print(f"  {table:<38} {len(rows):>5} features flattened"
          + (f"  ({skipped} skipped)" if skipped else ""))
    return rows, coverage


print("Flattening bronze vector layers:")
feature_rows = []
coverage_rows = []
for layer_config in VECTOR_LAYERS:
    layer_rows, layer_coverage = flatten_layer(layer_config)
    feature_rows.extend(layer_rows)
    coverage_rows.append(layer_coverage)

## 7. Catalogue-table coverage

STAC tables hold no geometry to flatten, but the report must be able to say which
configured sources returned nothing. An empty source is a **data gap**, not a zero.

In [ ]:
print("Checking catalogue (STAC metadata) tables:")
for table_name, source_name in CATALOG_LAYERS:
    coverage = {
        "run_id": RUN_ID,
        "source_name": source_name,
        "table_name": table_name,
        "feature_class": "stac_catalog",
        "status": "available",
        "record_count": 0,
        "note": None,
        "checked_at_utc": NOW_UTC,
    }
    try:
        count = read_bronze(table_name).count()
        coverage["record_count"] = int(count)
        if count == 0:
            coverage["status"] = "empty"
            coverage["note"] = "no catalogued items over this AOI"
        print(f"  {table_name:<38} {count:>5} items")
    except Exception as error:
        coverage["status"] = "unavailable"
        coverage["record_count"] = None
        coverage["note"] = f"bronze table could not be read: {str(error).splitlines()[0][:200]}"
        print(f"  {table_name:<38} UNAVAILABLE")
    coverage_rows.append(coverage)

## 8. Write the silver tables

Both tables are partitioned by `run_id` and written with dynamic partition overwrite,
so re-running one screening run replaces only that run and leaves earlier runs intact.
This is what makes the data agent's *"filter every query by the run identifier"*
instruction actually enforceable.

In [ ]:
def write_run_scoped(dataframe, table_name):
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
            .saveAsTable(table_name))
    except Exception as error:
        # A schema change cannot be applied in dynamic partition overwrite mode:
        # Delta rejects overwriteSchema with DELTA_OVERWRITE_SCHEMA_WITH_DYNAMIC_
        # PARTITION_OVERWRITE. Drop back to a static full overwrite, which replaces
        # every run, then restore dynamic mode for subsequent writes.
        print(f"  WARNING: {table_name} schema changed - replacing ALL runs. "
              f"({str(error).splitlines()[0][:120]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


ordered_feature_columns = [field.name for field in FEATURE_SCHEMA.fields]
feature_records = [
    tuple(row.get(column) for column in ordered_feature_columns) for row in feature_rows
]
features_df = spark.createDataFrame(feature_records, schema=FEATURE_SCHEMA)
write_run_scoped(features_df, "silver_source_features")

ordered_coverage_columns = [field.name for field in COVERAGE_SCHEMA.fields]
coverage_records = [
    tuple(row.get(column) for column in ordered_coverage_columns) for row in coverage_rows
]
coverage_df = spark.createDataFrame(coverage_records, schema=COVERAGE_SCHEMA)
write_run_scoped(coverage_df, "silver_source_coverage")

print(f"\nsilver_source_features : {features_df.count():,} rows")
print(f"silver_source_coverage : {coverage_df.count()} sources")

## 9. Verify

A quick look at what the report agent will be able to reason over.

In [ ]:
features = spark.read.table("silver_source_features").filter(F.col("run_id") == RUN_ID)

print("Features by class:")
features.groupBy("feature_class").agg(
    F.count(F.lit(1)).alias("features"),
    F.round(F.sum("area_km2"), 2).alias("total_area_km2"),
    F.round(F.min("distance_from_aoi_km"), 3).alias("nearest_km"),
).orderBy("feature_class").show(truncate=False)

print("Soil survey attribute fill rate (how much typed detail the agent actually gets):")
soil = features.filter(F.col("feature_class") == "soil_survey")
soil_total = soil.count()
if soil_total:
    fill = soil.select([
        F.round(F.count(F.col(column)) / F.lit(soil_total) * 100.0, 1).alias(column)
        for column in ["name", "unit_code", "drainage_class", "parent_material", "texture", "subgroup"]
    ])
    fill.show(truncate=False)
    print("Most common drainage classes:")
    (soil.groupBy("drainage_class")
         .agg(F.count(F.lit(1)).alias("features"),
              F.round(F.avg("soft_soil_score"), 3).alias("mean_soft_score"))
         .orderBy(F.desc("features")).show(10, truncate=False))
else:
    print("  no soil survey features over this AOI")

print("Source coverage:")
spark.read.table("silver_source_coverage").filter(F.col("run_id") == RUN_ID) \
    .select("table_name", "status", "record_count", "note") \
    .orderBy("table_name").show(30, truncate=60)